In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder
import warnings, os

warnings.filterwarnings("ignore")
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

### Load Data

In [86]:
df = pd.read_csv("D:/project/Global AI Adoption & Workforce Impact/Data/ai_company_adoption.csv")
print("Shape:", df.shape)

Shape: (150000, 43)


### Basic Checks

In [87]:
df.head()

,response_id,company_id,survey_year,quarter,country,region,industry,company_size,num_employees,annual_revenue_usd_millions,...,productivity_change_percent,jobs_displaced,jobs_created,reskilled_employees,revenue_growth_percent,cost_reduction_percent,innovation_score,customer_satisfaction,survey_source,data_collection_method
0,1,COMP-00001,2023,Q1,Italy,Europe,Education,Startup,57,48.31,...,2.65,1,1,3,2.52,9.45,53,5.20,WEF Survey,API Scrape
1,2,COMP-00001,2023,Q2,Italy,Europe,Education,Startup,57,48.31,...,5.77,2,2,5,4.77,0.00,51,6.98,McKinsey Report,Phone Interview
2,3,COMP-00001,2023,Q3,Italy,Europe,Education,Startup,57,48.31,...,6.94,3,3,2,12.87,9.74,40,4.12,Internal Corporate Survey,Research Compilation
3,4,COMP-00001,2023,Q4,Italy,Europe,Education,Startup,57,48.31,...,5.32,1,1,2,8.19,0.00,51,5.72,Internal Corporate Survey,Research Compilation
4,5,COMP-00001,2024,Q1,Italy,Europe,Education,Startup,57,48.31,...,6.32,2,3,6,11.30,9.02,43,6.31,McKinsey Report,Research Compilation


In [88]:
df.isnull().sum()

response_id                    0
company_id                     0
survey_year                    0
quarter                        0
country                        0
region                         0
industry                       0
company_size                   0
num_employees                  0
annual_revenue_usd_millions    0
company_founding_year          0
company_age                    0
company_age_group              0
ai_adoption_rate               0
ai_adoption_stage              0
years_using_ai                 0
ai_primary_tool                0
num_ai_tools_used              0
ai_use_case                    0
ai_projects_active             0
ai_training_hours              0
ai_budget_percentage           0
ai_maturity_score              0
ai_failure_rate                0
ai_investment_per_employee     0
regulatory_compliance_score    0
data_privacy_level             0
ai_ethics_committee            0
ai_risk_management_score       0
remote_work_percentage         0
employee_s

In [89]:
df.duplicated().sum()

np.int64(0)

In [90]:
rows_per_company = df["company_id"].value_counts()
print(f"Companies: {len(rows_per_company):,}   "
      f"\nRows/company: min={rows_per_company.min()} "
      f"\nmean={rows_per_company.mean():.1f} "
      f"\nmax={rows_per_company.max()}")

Companies: 10,000   
Rows/company: min=10 
mean=15.0 
max=16


## Type Fixes

In [91]:
quarter_order = ["Q1", "Q2", "Q3", "Q4"]
df["quarter"] = pd.Categorical(df["quarter"], categories=quarter_order, ordered=True)

In [92]:
df["has_ethics_committee"] = (df["ai_ethics_committee"] == "Yes").astype(int)

In [93]:
adoption_stage_order = {"none": 0, "pilot": 1, "partial": 2, "full": 3}
df["adoption_stage_ord"] = df["ai_adoption_stage"].map(adoption_stage_order)

In [94]:
privacy_order = {"Low": 0, "Medium": 1, "High": 2}
df["data_privacy_ord"] = df["data_privacy_level"].map(privacy_order)

## Feature Engineering

In [95]:
for col in ["annual_revenue_usd_millions", "num_employees", "ai_investment_per_employee"]:
    df[f"log_{col}"] = np.log1p(df[col])
    print(f"  log1p({col})  skew: {df[col].skew():.2f} → {df[f'log_{col}'].skew():.2f}")

  log1p(annual_revenue_usd_millions)  skew: 2.28 → 0.33
  log1p(num_employees)  skew: 2.28 → 0.47
  log1p(ai_investment_per_employee)  skew: 4.34 → -2.82


In [96]:
df["ai_intensity"] = df["ai_budget_percentage"] * df["ai_training_hours"]

In [97]:
df["roi_proxy"] = df["revenue_growth_percent"] + df["cost_reduction_percent"]

In [98]:
df["productivity_per_dollar"] = (
    df["productivity_change_percent"]
    / (df["ai_investment_per_employee"].replace(0, np.nan))
)

In [99]:
df["time_index"] = (df["survey_year"] - 2023) * 4 + df["quarter"].cat.codes

## Outlier Treatment

In [100]:
def iqr_bounds(series, k=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

In [101]:
skewed_cols = [
    "annual_revenue_usd_millions",
    "num_employees",
    "ai_investment_per_employee",
]

In [102]:
for col in skewed_cols:
    lo, hi = iqr_bounds(df[col])
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    # Winsorize at 1st/99th percentile (safe for all model types)
    p01, p99 = df[col].quantile(0.01), df[col].quantile(0.99)
    df[f"{col}_w"] = df[col].clip(lower=p01, upper=p99)
    print(f"  {col}: {n_out} IQR-3 outliers → winsorized col saved as {col}_w")

  annual_revenue_usd_millions: 26272 IQR-3 outliers → winsorized col saved as annual_revenue_usd_millions_w
  num_employees: 26174 IQR-3 outliers → winsorized col saved as num_employees_w
  ai_investment_per_employee: 5231 IQR-3 outliers → winsorized col saved as ai_investment_per_employee_w


### CORRELATION & MULTICOLLINEARITY

In [103]:
print("\n── Top feature correlations with key targets ──")
num_features = [
    "ai_maturity_score", "ai_budget_percentage", "ai_training_hours",
    "years_using_ai", "num_ai_tools_used", "ai_projects_active",
    "task_automation_rate", "time_saved_per_week", "ai_failure_rate",
    "regulatory_compliance_score", "ai_risk_management_score",
    "remote_work_percentage", "employee_satisfaction_score",
    "log_annual_revenue_usd_millions", "log_num_employees",
]

for target in ["revenue_growth_percent", "productivity_change_percent", "roi_proxy"]:
    corrs = df[num_features + [target]].corr()[target].drop(target).abs().sort_values(ascending=False)
    print(f"\n  Top 5 correlates of {target}:")
    for feat, r in corrs.head(5).items():
        print(f"    {feat:<45} r = {r:.3f}")


── Top feature correlations with key targets ──

  Top 5 correlates of revenue_growth_percent:
    ai_maturity_score                             r = 0.403
    ai_training_hours                             r = 0.348
    ai_budget_percentage                          r = 0.341
    task_automation_rate                          r = 0.327
    ai_failure_rate                               r = 0.322

  Top 5 correlates of productivity_change_percent:
    ai_maturity_score                             r = 0.737
    ai_training_hours                             r = 0.634
    ai_budget_percentage                          r = 0.604
    ai_failure_rate                               r = 0.594
    task_automation_rate                          r = 0.583

  Top 5 correlates of roi_proxy:
    ai_maturity_score                             r = 0.505
    task_automation_rate                          r = 0.495
    ai_training_hours                             r = 0.434
    ai_budget_percentage              

In [104]:
# Drop highly collinear features (r > 0.95)
corr_matrix = df[num_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
if high_corr:
    print(f"\n  High-collinearity cols to consider dropping: {high_corr}")
else:
    print("\n  No extreme collinearity found (r > 0.95).")


  No extreme collinearity found (r > 0.95).


## CLASS IMBALANCE REPORT

In [105]:

print("\n── Class imbalance (adoption stage) ──")
vc = df["ai_adoption_stage"].value_counts()
for stage, n in vc.items():
    print(f"  {stage:<10} {n:>7,}  ({n/len(df)*100:.1f}%)")
print("  Recommendation: use class_weight='balanced' or SMOTE for Module 1.")


── Class imbalance (adoption stage) ──
  partial     78,800  (52.5%)
  pilot       64,317  (42.9%)
  none         5,198  (3.5%)
  full         1,685  (1.1%)
  Recommendation: use class_weight='balanced' or SMOTE for Module 1.


In [106]:
drop_cols = [
    "company_founding_year",
    "company_age_group",       
    "survey_source",           
    "data_collection_method", 
]
df_clean = df.drop(columns=drop_cols)
df_clean.to_parquet("data/cleaned.parquet", index=False)

# EDA

In [107]:
sns.set_theme(style="whitegrid", palette="muted", font_scale=0.95)
fig = plt.figure(figsize=(18, 16))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

<Figure size 1800x1600 with 0 Axes>

In [108]:
# Adoption stage distribution
ax1 = fig.add_subplot(gs[0, 0])
vc.plot(kind="bar", ax=ax1, color=["#378ADD", "#97C459", "#888780", "#1D9E75"])
ax1.set_title("Adoption stage distribution")
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=20)

In [109]:
# Adoption rate by industry
ax3 = fig.add_subplot(gs[0, 2])
ind_means = df.groupby("industry")["ai_adoption_rate"].mean().sort_values()
ind_means.plot(kind="barh", ax=ax3, color="#7F77DD")
ax3.set_title("Mean adoption rate by industry")
ax3.set_xlabel("Adoption rate (%)")

Text(0.5, 0, 'Adoption rate (%)')

In [110]:
# Maturity score vs productivity (scatter)
ax4 = fig.add_subplot(gs[1, 0])
sample = df.sample(3000, random_state=42)
ax4.scatter(sample["ai_maturity_score"], sample["productivity_change_percent"],
            alpha=0.3, s=10, color="#1D9E75")
m, b, r, p, _ = stats.linregress(df["ai_maturity_score"], df["productivity_change_percent"])
x_line = np.linspace(df["ai_maturity_score"].min(), df["ai_maturity_score"].max(), 100)
ax4.plot(x_line, m * x_line + b, color="#D85A30", linewidth=1.5)
ax4.set_title(f"Maturity vs productivity  (r={r:.2f})")
ax4.set_xlabel("AI maturity score")
ax4.set_ylabel("Productivity change %")

Text(0, 0.5, 'Productivity change %')

In [111]:
# Revenue growth by company size
ax5 = fig.add_subplot(gs[1, 1])
df.boxplot(column="revenue_growth_percent", by="company_size", ax=ax5,
           boxprops=dict(color="#378ADD"), medianprops=dict(color="#D85A30"))
ax5.set_title("Revenue growth by company size")
ax5.set_xlabel("")
plt.sca(ax5)
plt.title("Revenue growth by company size")
fig.sca(ax5)

<Axes: title={'center': 'Revenue growth by company size'}>

In [112]:
# Ethics committee impact
ax6 = fig.add_subplot(gs[1, 2])
eth = df.groupby("ai_ethics_committee")[["revenue_growth_percent", "ai_failure_rate"]].mean()
eth.plot(kind="bar", ax=ax6, color=["#378ADD", "#D85A30"])
ax6.set_title("Ethics committee: avg outcomes")
ax6.set_xlabel("")
ax6.tick_params(axis="x", rotation=0)
ax6.legend(fontsize=8)

In [117]:
# Adoption rate trend over time
ax7 = fig.add_subplot(gs[2, 0])
trend = df.groupby("survey_year")["ai_adoption_rate"].mean()
trend.plot(ax=ax7, marker="o", color="#7F77DD", linewidth=2)
ax7.set_title("Adoption rate trend 2023–2026")
ax7.set_xlabel("Year")
ax7.set_ylabel("Mean adoption rate (%)")

Text(0, 0.5, 'Mean adoption rate (%)')

In [114]:
# Correlation heatmap (key features)
ax8 = fig.add_subplot(gs[2, 1:])
heat_cols = [
    "ai_maturity_score", "ai_budget_percentage", "ai_training_hours",
    "ai_failure_rate", "productivity_change_percent",
    "revenue_growth_percent", "cost_reduction_percent",
]
corr = df[heat_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.4, ax=ax8, annot_kws={"size": 8})
ax8.set_title("Correlation matrix (key features)")
ax8.tick_params(axis="x", rotation=30)
ax8.tick_params(axis="y", rotation=0)

In [115]:
plt.suptitle("AI Company Adoption — EDA Overview", fontsize=14, y=1.01)
plt.savefig("outputs/eda_report.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: outputs/eda_report.png")

Saved: outputs/eda_report.png
